# Diffusers (Stable Diffusion & image generation)

Hugging Face **`diffusers`** is the standard library for running and training diffusion
models — text-to-image, image-to-image, inpainting, video, audio. It is to diffusion what
`transformers` is to language models: a thin, consistent wrapper over a zoo of
architectures, with the awkward parts (schedulers, guidance, safety, memory) already
solved.

This library already covers the *theory* — [Diffusion Models](../08-architectures/diffusion-models.ipynb),
[Diffusion Transformers](../08-architectures/diffusion-transformer.ipynb),
[U-Net](../08-architectures/u-net.ipynb), [VAE](../08-architectures/vae.ipynb). This
notebook is the **practical** counterpart: the API, the knobs that matter, and the
arithmetic behind them worked out in NumPy so the mechanism is not a black box.

Model weights are a multi-gigabyte download, so the real pipeline call is gated behind
an environment variable. Everything else runs on CPU in seconds.

## 1. What & Why

A diffusion model generates by **iterative denoising**. Start from pure Gaussian noise
and repeatedly ask a network "what would this look like with slightly less noise?",
steering each step toward a text prompt. After 20–50 steps you have an image.

`diffusers` exists because assembling that from scratch means correctly wiring four
separate models (text encoder, U-Net or DiT, VAE, scheduler), each with its own
conventions about noise parameterisation, timestep spacing and latent scaling — and
getting any of them subtly wrong produces plausible-looking garbage rather than an error.

**Reach for it when:**

- You want to generate or edit images, video or audio with an open-weight model.
- You need control beyond a prompt — ControlNet, IP-Adapter, img2img, inpainting.
- You want to fine-tune (LoRA, DreamBooth, textual inversion) on your own images.
- You need the pipeline *components* individually, for research or a custom loop.

**Don't when:** you just want a hosted API and have no interest in local weights, or you
need a single classifier-style prediction — diffusion is a generative sampler, and an
expensive one.

## 2. Mental Model

**Sculpting from noise, with four collaborators.**

- The **VAE** is the workshop's scale model. Working on 512×512 pixels directly is
  ruinously expensive, so Stable Diffusion works in a 64×64×4 *latent* space and the VAE
  translates between the two. Encode at the start (for img2img), decode at the end.
- The **U-Net / DiT** is the sculptor. Given a noisy latent and a timestep, it predicts
  the noise to remove. This is the only learned "denoiser" and it is where the parameters
  are.
- The **text encoder** (CLIP or T5) turns the prompt into vectors the sculptor consults
  through cross-attention. It is the client's brief.
- The **scheduler** is the work plan: how many steps, how much noise to remove at each
  one, and how to turn the predicted noise into the next latent. It is pure mathematics —
  **no learned parameters** — which is why you can swap it after training and why the same
  weights give different results under DDIM, Euler or DPM-Solver.

The single most useful consequence: **the scheduler is a choice, not a property of the
model.** Most "this model needs 50 steps" folklore is really "the default scheduler needs
50 steps"; a better solver gets equivalent quality in 20.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Pipeline** | The assembled object (`StableDiffusionXLPipeline`, `FluxPipeline`, …). Holds the components and implements the sampling loop. |
| **Latent space** | The compressed space the diffusion actually happens in. SD1.5/SDXL use 4 channels at 1/8 resolution. |
| **Scheduler / sampler** | The discretised reverse process: DDPM, DDIM, Euler, DPM-Solver++, and so on. Swappable at inference. |
| **Inference steps** | How many denoising iterations. More is slower; quality saturates, typically 20–30 with a modern solver. |
| **Guidance scale (CFG)** | How hard to push toward the prompt. `pred = uncond + s·(cond − uncond)`. Typically 5–8. |
| **Classifier-free guidance** | The trick behind CFG: train with prompt dropout so one model gives both conditional and unconditional predictions. Costs 2× compute per step. |
| **Negative prompt** | The text used for the "unconditional" branch — so it is a *steer away from*, not a filter. |
| **`generator`** | The seeded RNG. Reproducibility comes from passing a `torch.Generator`, not from `torch.manual_seed` alone. |
| **Strength (img2img)** | How much of the input to destroy before re-denoising. 0 returns the input, 1 ignores it. |
| **LoRA / adapters** | Small fine-tuned weight deltas loaded onto a base model. See [LoRA](../03-llm-inference-training-optimization/lora-controlnet.ipynb). |
| **ControlNet** | A parallel network conditioning generation on structure — edges, depth, pose. |
| **Attention slicing / VAE tiling / CPU offload** | Memory-reduction switches that trade speed for fitting in less VRAM. |

## 4. Setup

The runnable examples use NumPy only. The real pipeline needs `torch` + `diffusers` and
a model download of several gigabytes, so it is gated behind `PRAXIS_RUN_HEAVY=1`.

In [1]:
# %pip install numpy
# For the gated example only:
# %pip install torch diffusers transformers accelerate safetensors

import os
import numpy as np

rng = np.random.default_rng(0)
RUN_HEAVY = os.getenv("PRAXIS_RUN_HEAVY") == "1"
print("numpy", np.__version__)
print("PRAXIS_RUN_HEAVY =", RUN_HEAVY, "- set to 1 to actually download weights and sample")

numpy 2.5.1
PRAXIS_RUN_HEAVY = False - set to 1 to actually download weights and sample


## 5. Worked Examples

### Example 1 — the noise schedule, which is what a scheduler *is*

Every diffusion model is defined by how much signal survives at each timestep. That is
the `alpha_bar` curve, and it explains why early steps change composition while late
steps only change texture.

In [2]:
T = 1000

def make_schedule(kind, T=T):
    if kind == "linear":                      # DDPM original
        betas = np.linspace(1e-4, 0.02, T)
    elif kind == "scaled_linear":             # Stable Diffusion's actual default
        betas = np.linspace(1e-4**0.5, 0.02**0.5, T) ** 2
    elif kind == "cosine":                    # improved DDPM
        s = 0.008
        t = np.linspace(0, 1, T + 1)
        ab = np.cos((t + s) / (1 + s) * np.pi / 2) ** 2
        ab = ab / ab[0]
        betas = np.clip(1 - ab[1:] / ab[:-1], 0, 0.999)
    return betas, np.cumprod(1.0 - betas)

print(f"{'timestep':>9} " + " ".join(f"{k:>16}" for k in ("linear", "scaled_linear", "cosine")))
print(f"{'':9} " + " ".join(f"{'signal retained':>16}" for _ in range(3)))
for t in (0, 100, 250, 500, 750, 999):
    row = []
    for kind in ("linear", "scaled_linear", "cosine"):
        _, ab = make_schedule(kind)
        row.append(f"{np.sqrt(ab[t]):16.4f}")
    print(f"{t:9d} " + " ".join(row))

print("\nsqrt(alpha_bar) is the fraction of the ORIGINAL image still present at step t.")
print("A noisy latent at step t is:  x_t = sqrt(ab)*x_0 + sqrt(1-ab)*noise")
print("\nNote how much earlier the linear schedule destroys the signal -- by t=500 it")
print("keeps 3% while cosine keeps 34%. That is why the cosine schedule spends more")
print("of its budget on the steps that carry actual content.")

 timestep           linear    scaled_linear           cosine
           signal retained  signal retained  signal retained
        0           0.9999           0.9999           1.0000
      100           0.9461           0.9855           0.9857
      250           0.7221           0.9055           0.9197
      500           0.2789           0.5756           0.7016
      750           0.0574           0.1953           0.3784
      999           0.0064           0.0271           0.0000

sqrt(alpha_bar) is the fraction of the ORIGINAL image still present at step t.
A noisy latent at step t is:  x_t = sqrt(ab)*x_0 + sqrt(1-ab)*noise

Note how much earlier the linear schedule destroys the signal -- by t=500 it
keeps 3% while cosine keeps 34%. That is why the cosine schedule spends more
of its budget on the steps that carry actual content.


### Example 2 — classifier-free guidance is just a vector extrapolation

CFG is the knob people turn most and understand least. It is one line, and seeing it as
extrapolation explains every artefact it causes.

In [3]:
# Two noise predictions from the same model: one that saw the prompt, one that did not.
noise_uncond = np.array([0.20, -0.10, 0.05, 0.30])
noise_cond   = np.array([0.35,  0.05, 0.00, 0.25])

print(f"{'scale':>6} {'guided prediction':>34} {'norm':>8} {'note'}")
for s in (0.0, 1.0, 3.0, 7.5, 15.0, 30.0):
    guided = noise_uncond + s * (noise_cond - noise_uncond)
    note = ("ignores the prompt entirely" if s == 0 else
            "the raw conditional prediction" if s == 1 else
            "typical range" if s <= 8 else
            "over-saturated, over-contrasty")
    print(f"{s:6.1f} {str(np.round(guided, 3)):>34} {np.linalg.norm(guided):8.3f}  {note}")

print("\nAt s=1 you get exactly the conditional prediction. Above that you are")
print("EXTRAPOLATING past it, along the direction 'what the prompt added'.")
print("\nThat is why high CFG blows out contrast and saturation: the norm of the")
print("prediction grows without bound, and the sampler is being pushed to a place")
print("no training example ever occupied.")
print("\nIt is also why CFG costs 2x: every step runs the model twice, once with the")
print("prompt and once with the negative prompt, to get both vectors.")

 scale                  guided prediction     norm note
   0.0          [ 0.2  -0.1   0.05  0.3 ]    0.377  ignores the prompt entirely
   1.0              [0.35 0.05 0.   0.25]    0.433  the raw conditional prediction
   3.0          [ 0.65  0.35 -0.1   0.15]    0.760  typical range
   7.5      [ 1.325  1.025 -0.325 -0.075]    1.708  typical range
  15.0          [ 2.45  2.15 -0.7  -0.45]    3.364  over-saturated, over-contrasty
  30.0          [ 4.7   4.4  -1.45 -1.2 ]    6.708  over-saturated, over-contrasty

At s=1 you get exactly the conditional prediction. Above that you are
EXTRAPOLATING past it, along the direction 'what the prompt added'.

That is why high CFG blows out contrast and saturation: the norm of the
prediction grows without bound, and the sampler is being pushed to a place
no training example ever occupied.

It is also why CFG costs 2x: every step runs the model twice, once with the
prompt and once with the negative prompt, to get both vectors.


### Example 3 — a complete DDIM sampling loop

The whole reverse process in about fifteen lines, with a toy 1-D "model" so it runs
instantly. The structure here is exactly what `pipe()` does internally.

In [4]:
betas, alpha_bar = make_schedule("scaled_linear")

# The "dataset": three 4-D vectors standing in for images. A real model learns this
# distribution; here we can compute the ideal denoiser in closed form, which makes the
# sampler's behaviour attributable to the sampler rather than to model error.
PATTERNS = np.array([[1.0, -0.5, 0.8, -1.0],
                     [-0.9, 1.0, -0.7, 0.6],
                     [0.2, 0.9, 1.0, 0.4]])

def predict_x0(x, t, conditional=True):
    '''The ideal denoiser: E[x_0 | x_t] under the mixture above.

    Conditional -> the posterior over patterns, which sharpens as t falls.
    Unconditional -> no information, so the mean of the dataset.
    '''
    ab = alpha_bar[t]
    if not conditional:
        return PATTERNS.mean(axis=0)
    logits = -np.sum((x - np.sqrt(ab) * PATTERNS) ** 2, axis=1) / (2 * (1 - ab))
    w = np.exp(logits - logits.max())
    return (w / w.sum()) @ PATTERNS

def predict_noise(x, t, conditional):
    ab = alpha_bar[t]
    return (x - np.sqrt(ab) * predict_x0(x, t, conditional)) / np.sqrt(1 - ab)

def ddim_sample(steps=20, guidance=1.0, seed=0):
    x = np.random.default_rng(seed).standard_normal(4)          # pure noise
    timesteps = np.linspace(T - 1, 0, steps).astype(int)

    for i, t in enumerate(timesteps):
        # classifier-free guidance: two predictions, extrapolate between them
        eps_uncond = predict_noise(x, t, conditional=False)
        eps_cond = predict_noise(x, t, conditional=True)
        eps = eps_uncond + guidance * (eps_cond - eps_uncond)

        ab_t = alpha_bar[t]
        x0_pred = (x - np.sqrt(1 - ab_t) * eps) / np.sqrt(ab_t)   # implied clean sample
        if i + 1 < len(timesteps):
            ab_prev = alpha_bar[timesteps[i + 1]]
            x = np.sqrt(ab_prev) * x0_pred + np.sqrt(1 - ab_prev) * eps   # DDIM step
        else:
            x = x0_pred
    return x

off_manifold = lambda v: min(np.linalg.norm(v - p) for p in PATTERNS)

print("step count (guidance = 1):")
for steps in (1, 2, 3, 5, 10, 20, 50):
    out = ddim_sample(steps=steps)
    print(f"  {steps:3d} -> {str(np.round(out, 3)):>26}  distance to nearest real "
          f"sample {off_manifold(out):.4f}")

print("\nOne step lands between the patterns -- the average of a distribution is not a")
print("member of it, which is precisely why one-step generation looks blurry. From two")
print("steps on, this toy has converged exactly.")

print("\nthe seed chooses the mode (30 steps, guidance = 1):")
for sd in range(6):
    out = ddim_sample(steps=30, seed=sd)
    print(f"  seed {sd} -> pattern {int(np.argmin([np.linalg.norm(out - p) for p in PATTERNS]))}")

print("\nguidance (30 steps, seed 0):")
for g in (0.0, 1.0, 2.0, 4.0, 8.0, 15.0):
    out = ddim_sample(steps=30, guidance=g)
    print(f"  g={g:5.1f}  |x| = {np.linalg.norm(out):7.3f}   distance off the data "
          f"manifold {off_manifold(out):.4f}")

step count (guidance = 1):
    1 -> [ 0.111  0.461  0.378 -0.007]  distance to nearest real sample 0.8675
    2 ->          [0.2 0.9 1.  0.4]  distance to nearest real sample 0.0000
    3 ->          [0.2 0.9 1.  0.4]  distance to nearest real sample 0.0000
    5 ->          [0.2 0.9 1.  0.4]  distance to nearest real sample 0.0000
   10 ->          [0.2 0.9 1.  0.4]  distance to nearest real sample 0.0000
   20 ->          [0.2 0.9 1.  0.4]  distance to nearest real sample 0.0000
   50 ->          [0.2 0.9 1.  0.4]  distance to nearest real sample 0.0000

One step lands between the patterns -- the average of a distribution is not a
member of it, which is precisely why one-step generation looks blurry. From two
steps on, this toy has converged exactly.

the seed chooses the mode (30 steps, guidance = 1):
  seed 0 -> pattern 2
  seed 1 -> pattern 0
  seed 2 -> pattern 0
  seed 3 -> pattern 0
  seed 4 -> pattern 2
  seed 5 -> pattern 1

guidance (30 steps, seed 0):
  g=  0.0  |x| =   0.6

The guidance sweep is the geometry behind every over-cooked CFG image. At `g = 1` the
sampler lands **exactly** on a real data point. At `g = 0` it lands on the dataset mean —
a valid average, not a valid sample. Above 1 the distance from the data manifold grows
roughly linearly and `|x|` grows with it: guidance is extrapolation, and past a point you
are extrapolating into a region the model never saw. In pixel space that reads as blown
highlights, posterised colour and over-contrast.

**What this toy does and does not show.** It reproduces the DDIM loop exactly as written
in the library, the sharpening posterior, the seed's role in mode selection, and CFG's
extrapolation geometry. It cannot show the *quality-versus-diversity* trade-off that
makes moderate guidance genuinely useful on real models — that needs a real conditional
distribution, whereas this "dataset" is three point masses with nothing to trade. Take
the mechanism from here and the tuning ranges from Example 4.

### Example 4 — the real `diffusers` API

Gated behind `PRAXIS_RUN_HEAVY=1`. The code below is what you would actually write; the
comments mark the places where the defaults bite.

In [5]:
PIPELINE_CODE = '''
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,        # fp16 on CUDA/MPS; use float32 on CPU
    variant="fp16",
    use_safetensors=True,
)
pipe = pipe.to("cuda")

# The scheduler is a free choice -- swapping it is the cheapest quality/speed win.
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Reproducibility comes from an explicit generator, NOT torch.manual_seed().
generator = torch.Generator(device="cuda").manual_seed(42)

image = pipe(
    prompt="a lighthouse on a basalt cliff, storm light, photographic",
    negative_prompt="blurry, low contrast, watermark",   # the CFG unconditional branch
    num_inference_steps=25,          # 20-30 is plenty with DPM-Solver++
    guidance_scale=6.5,              # >10 over-saturates (Example 2)
    height=1024, width=1024,         # SDXL is trained at 1024; 512 degrades badly
    generator=generator,
).images[0]

image.save("out.png")

# --- memory switches, in increasing order of desperation ---
pipe.enable_attention_slicing()      # small speed cost
pipe.enable_vae_tiling()             # for very large outputs
pipe.enable_model_cpu_offload()      # big speed cost, big VRAM saving

# --- LoRA adapters ---
pipe.load_lora_weights("path/or/hub-id", adapter_name="style")
pipe.set_adapters(["style"], adapter_weights=[0.8])
'''

if RUN_HEAVY:
    import torch
    from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

    device = "cuda" if torch.cuda.is_available() else (
        "mps" if torch.backends.mps.is_available() else "cpu")
    dtype = torch.float16 if device == "cuda" else torch.float32
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=dtype, use_safetensors=True).to(device)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    gen = torch.Generator(device=device).manual_seed(42)
    img = pipe(prompt="a lighthouse on a basalt cliff, storm light",
               num_inference_steps=25, guidance_scale=6.5, generator=gen).images[0]
    img.save("out.png")
    print(f"wrote out.png ({img.size[0]}x{img.size[1]}) on {device}")
else:
    print("PRAXIS_RUN_HEAVY is not set - showing the pipeline code without downloading:")
    print(PIPELINE_CODE)

PRAXIS_RUN_HEAVY is not set - showing the pipeline code without downloading:

import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,        # fp16 on CUDA/MPS; use float32 on CPU
    variant="fp16",
    use_safetensors=True,
)
pipe = pipe.to("cuda")

# The scheduler is a free choice -- swapping it is the cheapest quality/speed win.
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

# Reproducibility comes from an explicit generator, NOT torch.manual_seed().
generator = torch.Generator(device="cuda").manual_seed(42)

image = pipe(
    prompt="a lighthouse on a basalt cliff, storm light, photographic",
    negative_prompt="blurry, low contrast, watermark",   # the CFG unconditional branch
    num_inference_steps=25,          # 20-30 is plenty with DPM-Solver++
    guidance_scale=6.5,             

## 6. Gotchas & Pitfalls

- **`torch.manual_seed()` does not make a pipeline reproducible.** Pass an explicit
  `torch.Generator` with the right `device`. Results also differ between CPU and GPU for
  the same seed — device-independent reproducibility is not on offer.
- **fp16 on CPU.** `torch_dtype=torch.float16` on CPU is either an error or extremely
  slow. Use `float32` on CPU and MPS unless you have checked.
- **Generating at the wrong resolution.** SD 1.5 is trained at 512, SDXL at 1024.
  Asking SDXL for 512×512 gives noticeably degraded output — this is a training-
  distribution issue, not a bug.
- **Cranking guidance for "better prompt adherence".** Example 2: past ~10 you are
  extrapolating well outside the training distribution, and you get burnt highlights and
  posterised colour. If a prompt is not being followed, fix the prompt or the model.
- **Expecting the negative prompt to be a filter.** It replaces the unconditional
  branch, so it defines the direction you are moving *away from*. Putting a concept there
  can, at high guidance, make it more prominent.
- **A black or green output image.** Classic symptoms: the NSFW safety checker triggered
  (returns a black image), or fp16 numerical overflow in the VAE (some SDXL VAEs need
  `madebyollin/sdxl-vae-fp16-fix` or fp32 for the VAE alone).
- **Leaving the default scheduler.** The config default is rarely the best sampler. A
  `DPMSolverMultistepScheduler` swap is one line and typically halves the steps needed.
- **Loading pipelines in a loop.** `from_pretrained` is slow and memory-hungry. Load
  once; use `DiffusionPipeline.from_pipe()` to share components between pipelines
  (text2img and img2img) without a second copy in VRAM.
- **Licence and provenance.** Model cards carry real restrictions (OpenRAIL, SDXL's
  terms, Flux's non-commercial variants), and the training data is a live legal question.
  Check the card before shipping output commercially.

## 7. When to Use vs Alternatives

| Need | Reach for |
|---|---|
| Open-weight image/video generation, locally | **`diffusers`** |
| Maximum speed on NVIDIA, production serving | `diffusers` + TensorRT / `torch.compile`, or [TensorRT-LLM](../11-devops-mlops-infra/model-serving-libraries/tensorrt-llm.ipynb)-style compiled pipelines |
| A node-graph UI, rapid visual experimentation | ComfyUI or Automatic1111 — both sit on `diffusers`/`transformers` underneath |
| Hosted, no infrastructure | A provider API — no weights, no VRAM, per-image cost |
| Understanding the maths | [Diffusion Models](../08-architectures/diffusion-models.ipynb) and [DiT](../08-architectures/diffusion-transformer.ipynb) |
| Fine-tuning on your own images | `diffusers` training scripts — LoRA, DreamBooth, textual inversion |

**How it compares to the GUI tools.** ComfyUI and Automatic1111 are better for
interactive exploration; `diffusers` is better when generation is a step inside a
program — a batch job, a service, a training loop, an evaluation harness. They are not
really competitors, and ComfyUI can load the same `safetensors` weights.

Within this library, `diffusers` is the practical sibling of the architecture notebooks:
read [Diffusion Models](../08-architectures/diffusion-models.ipynb) for *why* the reverse
process is what it is, and come here to run it. For the video case see
[Video Diffusion Models](../08-architectures/video-diffusion.ipynb), which uses the same
library.

## 8. Resources

- [Diffusers documentation](https://huggingface.co/docs/diffusers/index) — the reference; the "Effective and efficient diffusion" and "Load schedulers" pages are the highest-value starting points.
- [Diffusers GitHub repository](https://github.com/huggingface/diffusers) — includes the `examples/` training scripts for LoRA, DreamBooth and textual inversion.
- [High-Resolution Image Synthesis with Latent Diffusion Models](https://arxiv.org/abs/2112.10752) — the Stable Diffusion paper; the VAE-latent argument from the Mental Model section.
- [Classifier-Free Diffusion Guidance](https://arxiv.org/abs/2207.12598) — Ho & Salimans; Example 2, developed properly.
- [Denoising Diffusion Implicit Models](https://arxiv.org/abs/2010.02502) — DDIM, the deterministic sampler implemented in Example 3.
- [DPM-Solver++](https://arxiv.org/abs/2211.01095) — the fast solver worth switching to; why 20 steps is enough.
- [The Hugging Face Diffusion Models Course](https://github.com/huggingface/diffusion-models-class) — free, notebook-based, and a good bridge from this notebook to training your own.